In [17]:
import warnings
warnings.filterwarnings(action='ignore')
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors

# for descriptors calculation
import datamol as dm
from molfeat.calc.descriptors import RDKitDescriptors2D, RDKitDescriptors3D
from molfeat.trans import MoleculeTransformer

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np
import pickle
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from molfeat_padel.calc import PadelDescriptors


from rdkit import Chem
from rdkit.Chem import AllChem

from function import embed_optimize

In [18]:
pipe = make_pipeline(StandardScaler(), PCA(n_components=25))

In [2]:
with open('X_padel_dropped.pickle', 'rb') as inp:
    X = pickle.load(inp)

In [20]:
X = pipe.fit_transform(X)

In [21]:
tranpX = np.transpose(X)
xtx = np.dot(tranpX, X)
invxtx = np.linalg.pinv(xtx)

In [27]:
warning_level = 3*(X.shape[1]/X.shape[0])
print(warning_level)

0.40540540540540543


In [24]:
smi = 'Cc1ccccc1'
mol = embed_optimize(smi)

In [25]:
desc_calc = PadelDescriptors()
with dm.without_rdkit_log():
    descs = pd.DataFrame(desc_calc(mol), index = desc_calc.columns).transpose()
    
with open('imputer.pickle', 'rb') as inp:
    imputer = pickle.load(inp) 

descs = pd.DataFrame(imputer.transform(descs), columns = descs.columns)

with open('features_to_drop_Padel.pickle', 'rb') as inp:
        features_to_drop = pickle.load(inp)

descs = descs.drop(columns = features_to_drop)

In [28]:
descs_transformed = pipe.transform(descs)

In [31]:
leverage = np.dot(np.dot(descs_transformed, invxtx), np.transpose(descs_transformed))[0][0]
print(leverage)

0.0801896435195205


In [32]:
with open('app_domain.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx, 'warning':warning_level, 'pipe':pipe}, out)